# Basic Chatbot - HTMX v4 Version

This is the htmx v4 implementation of the basic chatbot example.

## Key Changes from htmx v2 to v4

| htmx v2 | htmx v4 |
|---------|---------|
| `FastHTML(hdrs=hdrs)` | `FastHTML(hdrs=hdrs, htmx=False, htmx4=True)` |
| Default htmx v2 headers | htmx v4 headers with `metaCharacter="-"` |

The basic chatbot works almost identically in htmx v4. The main change is just the constructor parameters. Features like `hx_post`, `hx_target`, `hx_swap`, and `hx_swap_oob` all work the same way.

In [2]:
from fasthtml.common import *
from fasthtml.jupyter import *
from claudette import *

In [7]:
# Set up the app, including daisyui and tailwind for the chat component
hdrs = (picolink, Script(src="https://cdn.tailwindcss.com"),
    Link(rel="stylesheet", href="https://cdn.jsdelivr.net/npm/daisyui@4.11.1/dist/full.min.css"))
app = FastHTML(hdrs=hdrs, htmx=False, htmx4=True, cls="p-4 max-w-lg mx-auto")

# Set up a chat model (https://claudette.answer.ai/)
cli = Client(models[1])
sp = "You are a helpful and concise assistant."

In [8]:
# Chat message component (renders a chat bubble)
def ChatMessage(msg, user):
    bubble_class = "chat-bubble-primary" if user else 'chat-bubble-secondary'
    chat_class = "chat-end" if user else 'chat-start'
    return Div(cls=f"chat {chat_class}")(
               Div('user' if user else 'assistant', cls="chat-header"),
               Div(msg, cls=f"chat-bubble {bubble_class}"),
               Hidden(msg, name="messages")
           )

# The input field for the user message. Also used to clear the
# input field after sending a message via an OOB swap
def ChatInput():
    return Input(name='msg', id='msg-input', placeholder="Type a message",
                 cls="input input-bordered w-full", hx_swap_oob='true')

In [11]:
# The main screen
@app.get
def index():
    page = Form(hx_post=send, hx_target="#chatlist", hx_swap="beforeend")(
           Div(id="chatlist", cls="chat-box h-[73vh] overflow-y-auto"),
               Div(cls="flex space-x-2 mt-2")(
                   Group(ChatInput(), Button("Send", cls="btn btn-primary"))
               )
           )
    return Titled('Chatbot Demo', page)

# Handle the form submission
@app.post
def send(msg:str, messages:list[str]=None):
    if not messages: messages = []
    messages.append(msg.rstrip())
    r = contents(cli(messages, sp=sp)) # get response from chat model
    return (ChatMessage(msg, True),    # The user's message
            ChatMessage(r.rstrip(), False), # The chatbot's response
            ChatInput()) # And clear the input field via an OOB swap

In [ ]:
srv = JupyUvi(app)